# U-10 (items 1-4): SVI-23 (Yeung & Mintzer)

This notebook contains implementation of items 1-11 for U-10.
Each item is separated into its own section.


## Preparation: load image `4/goldhill.tif`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

img_path = Path('4/goldhill.tif')
if not img_path.exists():
    raise FileNotFoundError(f'File not found: {img_path}')

C = np.array(Image.open(img_path).convert('L'), dtype=np.uint8)
H, W = C.shape

print(f'Container: {img_path}')
print(f'Shape: {C.shape}, dtype: {C.dtype}, min/max: {C.min()}/{C.max()}')

plt.figure(figsize=(5, 5))
plt.imshow(C, cmap='gray', vmin=0, vmax=255)
plt.title('Original container C (goldhill)')
plt.axis('off')
plt.show()


## Item 1. `yeung_genmapping`: generate mapping $\mu:[0,255]\to\{0,1\}$

Generate a pseudo-random binary mapping vector of length 256 from key `seed`.


In [ ]:
def yeung_genmapping(seed: int) -> np.ndarray:
    """Generate mapping mu by key seed (formula 7.16)."""
    rng = np.random.default_rng(seed)
    mapping = rng.integers(0, 2, size=256, dtype=np.uint8)

    # Ensure both values 0 and 1 are present.
    if mapping.min() == mapping.max():
        mapping[0] = 0
        mapping[1] = 1
    return mapping


seed = 2026
mapping = yeung_genmapping(seed)
print('mapping shape:', mapping.shape)
print('Count of zeros/ones:', int((mapping == 0).sum()), int((mapping == 1).sum()))
print('First 32 elements:', mapping[:32])


## Item 2. Start `yeung_extract`: compute matrix $\mu(C^W)$

Load a logo, build binary watermark template `Wr`, and implement the first extraction step `uCW = mu(CW)`.


In [ ]:
def to_binary_logo(arr: np.ndarray, threshold: int = 128) -> np.ndarray:
    """Threshold binarization of logo image (0/1)."""
    return (arr >= threshold).astype(np.uint8)


def resize_binary_logo(logo_bin: np.ndarray, shape_hw: tuple[int, int]) -> np.ndarray:
    """Resize binary logo to shape_hw=(H,W) without smoothing."""
    img = Image.fromarray((logo_bin * 255).astype(np.uint8), mode='L')
    img = img.resize((shape_hw[1], shape_hw[0]), resample=Image.Resampling.NEAREST)
    return (np.array(img, dtype=np.uint8) >= 128).astype(np.uint8)


def yeung_extract_uCW(CW: np.ndarray, mapping: np.ndarray) -> np.ndarray:
    """Initial extraction step: uCW = mu(CW)."""
    return mapping[CW.astype(np.uint8)]


# Logo file (change path if needed).
logo_path = Path('2/mickey.tif')
if not logo_path.exists():
    raise FileNotFoundError(f'Logo file not found: {logo_path}')

logo_img = np.array(Image.open(logo_path).convert('L'), dtype=np.uint8)
logo_bin = to_binary_logo(logo_img)

# Choose watermark template size Wr (M1 x M2).
M1, M2 = 64, 64
Wr = resize_binary_logo(logo_bin, (M1, M2))

# Extract mu(C) from original container (no embedding), item 2.
uC = yeung_extract_uCW(C, mapping)

print(f'Logo: {logo_path}, source size: {logo_img.shape}, Wr size: {Wr.shape}')

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(logo_bin, cmap='gray', vmin=0, vmax=1)
ax[0].set_title('Binary logo')
ax[0].axis('off')

ax[1].imshow(Wr, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Template Wr (64x64)')
ax[1].axis('off')

ax[2].imshow(uC, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('uC = mu(C)')
ax[2].axis('off')

plt.tight_layout()
plt.show()


## Item 3. Simplified `yeung_embed` (without error diffusion)

Implement simplified embedding: if current bit does not match target bit, replace pixel with a smaller admissible value (fallback to nearest upper value if needed).


In [ ]:
def tile_wr(Wr: np.ndarray, shape_hw: tuple[int, int]) -> np.ndarray:
    """Periodic tiling of Wr across full image."""
    h, w = shape_hw
    m1, m2 = Wr.shape
    yy = np.arange(h) % m1
    xx = np.arange(w) % m2
    return Wr[np.ix_(yy, xx)].astype(np.uint8)


def build_search_luts(mapping: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """LUT for nearest admissible levels: lower and upper."""
    lower = np.full((256, 2), -1, dtype=np.int16)
    upper = np.full((256, 2), -1, dtype=np.int16)

    for c in range(256):
        for bit in (0, 1):
            lo = np.where(mapping[: c + 1] == bit)[0]
            hi = np.where(mapping[c:] == bit)[0]
            if lo.size:
                lower[c, bit] = lo[-1]
            if hi.size:
                upper[c, bit] = c + hi[0]
    return lower, upper


def yeung_embed(C: np.ndarray, Wr: np.ndarray, mapping: np.ndarray) -> np.ndarray:
    """Simplified Yeung-Mintzer embedding without error diffusion."""
    C = C.astype(np.uint8)
    target = tile_wr(Wr, C.shape)
    current = mapping[C]

    CW = C.copy().astype(np.int16)
    mismatch = current != target

    lower, upper = build_search_luts(mapping)

    r, c = np.where(mismatch)
    if r.size:
        src_vals = C[r, c]
        bits = target[r, c]
        v = lower[src_vals, bits]

        # If no lower value exists, use nearest upper one.
        miss = v < 0
        if np.any(miss):
            v[miss] = upper[src_vals[miss], bits[miss]]

        CW[r, c] = v

    return np.clip(CW, 0, 255).astype(np.uint8)


CW_simple = yeung_embed(C, Wr, mapping)
changed_ratio = np.mean(CW_simple != C)

print('Changed pixel ratio:', float(changed_ratio))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(CW_simple, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('CW (simplified embedding)')
ax[0].axis('off')

ax[1].imshow((CW_simple != C).astype(np.uint8), cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Changed-pixel map')
ax[1].axis('off')

plt.tight_layout()
plt.show()


## Item 4. Complete `yeung_extract`, embedding, PSNR and extraction error rate

Finalize extraction: return change mask `E` and `uCW = mu(CW)`. Then compute PSNR and BER relative to expected template `Wr`.


In [ ]:
def yeung_extract(CW: np.ndarray, mapping: np.ndarray, Wr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Extraction: E (change mask) and uCW = mu(CW)."""
    uCW = yeung_extract_uCW(CW, mapping)
    expected = tile_wr(Wr, CW.shape)
    E = (uCW != expected).astype(np.uint8)
    return E, uCW


def psnr(x: np.ndarray, y: np.ndarray) -> float:
    x = x.astype(np.float64)
    y = y.astype(np.float64)
    mse = np.mean((x - y) ** 2)
    if mse == 0:
        return float('inf')
    return float(10.0 * np.log10((255.0 ** 2) / mse))


def bit_error_rate(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean(a.astype(np.uint8) != b.astype(np.uint8)))


E, uCW = yeung_extract(CW_simple, mapping, Wr)
expected = tile_wr(Wr, C.shape)

psnr_val = psnr(C, CW_simple)
ber_val = bit_error_rate(uCW, expected)

print(f'PSNR(C, CW): {psnr_val:.4f} dB')
print(f'Bit error rate (BER): {ber_val:.6f}')
print(f'Fraction of mismatches E=1: {E.mean():.6f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(expected, cmap='gray', vmin=0, vmax=1)
ax[0].set_title('Expected template Wr~')
ax[0].axis('off')

ax[1].imshow(uCW, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Extracted uCW = mu(CW)')
ax[1].axis('off')

ax[2].imshow(E, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('Mask E (1 = mismatch)')
ax[2].axis('off')

plt.tight_layout()
plt.show()


## Item 5. Full nearest-value search in `yeung_embed`

Implement full nearest-value search `v` such that `mu(v)=w` and compare with simplified embedding from item 3.


In [ ]:
def yeung_embed_full(C: np.ndarray, Wr: np.ndarray, mapping: np.ndarray) -> np.ndarray:
    """Full embedding: choose nearest v such that mu(v)=target bit."""
    C = C.astype(np.uint8)
    target = tile_wr(Wr, C.shape)
    current = mapping[C]
    CW = C.copy().astype(np.int16)

    mismatch = current != target
    if not np.any(mismatch):
        return C.copy()

    lower, upper = build_search_luts(mapping)
    r, c = np.where(mismatch)
    src = C[r, c]
    bits = target[r, c]

    lo = lower[src, bits]
    hi = upper[src, bits]

    dlo = np.where(lo >= 0, src.astype(np.int16) - lo, 10_000)
    dhi = np.where(hi >= 0, hi - src.astype(np.int16), 10_000)

    pick_hi = dhi < dlo
    v = lo.copy()
    v[pick_hi] = hi[pick_hi]

    fallback = v < 0
    if np.any(fallback):
        v[fallback] = hi[fallback]

    CW[r, c] = v
    return np.clip(CW, 0, 255).astype(np.uint8)


CW_full = yeung_embed_full(C, Wr, mapping)
E_full, uCW_full = yeung_extract(CW_full, mapping, Wr)
expected = tile_wr(Wr, C.shape)

psnr_simple = psnr(C, CW_simple)
ber_simple = bit_error_rate(yeung_extract_uCW(CW_simple, mapping), expected)

psnr_full = psnr(C, CW_full)
ber_full = bit_error_rate(uCW_full, expected)

print('Comparison: simplified vs full embedding:')
print(f'  simple: PSNR={psnr_simple:.4f} dB, BER={ber_simple:.6f}')
print(f'  full  : PSNR={psnr_full:.4f} dB, BER={ber_full:.6f}')

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(CW_simple, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('CW simple')
ax[0].axis('off')

ax[1].imshow(CW_full, cmap='gray', vmin=0, vmax=255)
ax[1].set_title('CW full (nearest search)')
ax[1].axis('off')

ax[2].imshow(np.abs(CW_full.astype(np.int16)-CW_simple.astype(np.int16)), cmap='hot')
ax[2].set_title('|CW_full - CW_simple|')
ax[2].axis('off')

plt.tight_layout()
plt.show()


## Item 6. Localized changes: blur fragment

Apply local blur to watermarked image and check how extraction localizes modified area.


In [ ]:
from PIL import ImageFilter


def blur_fragment(img: np.ndarray, top: int, left: int, h: int, w: int, radius: float = 2.0) -> tuple[np.ndarray, np.ndarray]:
    out = img.copy()
    mask = np.zeros_like(img, dtype=np.uint8)

    roi = out[top:top+h, left:left+w]
    roi_pil = Image.fromarray(roi, mode='L').filter(ImageFilter.GaussianBlur(radius=radius))
    out[top:top+h, left:left+w] = np.array(roi_pil, dtype=np.uint8)
    mask[top:top+h, left:left+w] = 1
    return out, mask


CW_blur, tamper_blur = blur_fragment(CW_full, top=170, left=180, h=120, w=120, radius=2.5)
E_blur, u_blur = yeung_extract(CW_blur, mapping, Wr)

print(f'PSNR(CW_full, CW_blur): {psnr(CW_full, CW_blur):.4f} dB')
print(f'Fraction E=1 after blur: {E_blur.mean():.6f}')

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].imshow(CW_blur, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('CW after blur')
ax[0].axis('off')

ax[1].imshow(tamper_blur, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Ground-truth tamper area')
ax[1].axis('off')

ax[2].imshow(E_blur, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('Extracted mask E')
ax[2].axis('off')

ax[3].imshow((E_blur & tamper_blur).astype(np.uint8), cmap='gray', vmin=0, vmax=1)
ax[3].set_title('Intersection: E & GT')
ax[3].axis('off')

plt.tight_layout()
plt.show()


## Item 7. Localized changes: replace region with another region from same image

Replace one region with another region from the same watermarked image and analyze extracted mask.


In [ ]:
def replace_with_internal_patch(img: np.ndarray,
                                dst_top: int, dst_left: int,
                                src_top: int, src_left: int,
                                h: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    out = img.copy()
    mask = np.zeros_like(img, dtype=np.uint8)
    patch = out[src_top:src_top+h, src_left:src_left+w].copy()
    out[dst_top:dst_top+h, dst_left:dst_left+w] = patch
    mask[dst_top:dst_top+h, dst_left:dst_left+w] = 1
    return out, mask


CW_swap_internal, tamper_internal = replace_with_internal_patch(
    CW_full,
    dst_top=320, dst_left=80,
    src_top=60, src_left=340,
    h=96, w=96,
)
E_internal, u_internal = yeung_extract(CW_swap_internal, mapping, Wr)

print(f'PSNR(CW_full, CW_swap_internal): {psnr(CW_full, CW_swap_internal):.4f} dB')
print(f'Fraction E=1 after internal replacement: {E_internal.mean():.6f}')

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].imshow(CW_swap_internal, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('CW after internal replacement')
ax[0].axis('off')

ax[1].imshow(tamper_internal, cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Ground-truth tamper area')
ax[1].axis('off')

ax[2].imshow(E_internal, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('Extracted mask E')
ax[2].axis('off')

ax[3].imshow((E_internal & tamper_internal).astype(np.uint8), cmap='gray', vmin=0, vmax=1)
ax[3].set_title('Intersection: E & GT')
ax[3].axis('off')

plt.tight_layout()
plt.show()


## Item 8. Localized changes: replace region with patch from another image

Replace region with patch from another image and evaluate localization performance.


In [ ]:
def replace_with_external_patch(img: np.ndarray,
                                donor: np.ndarray,
                                dst_top: int, dst_left: int,
                                src_top: int, src_left: int,
                                h: int, w: int) -> tuple[np.ndarray, np.ndarray]:
    out = img.copy()
    mask = np.zeros_like(img, dtype=np.uint8)
    donor_patch = donor[src_top:src_top+h, src_left:src_left+w].copy()
    out[dst_top:dst_top+h, dst_left:dst_left+w] = donor_patch
    mask[dst_top:dst_top+h, dst_left:dst_left+w] = 1
    return out, mask


donor_path = Path('2/house.bmp')
donor = np.array(Image.open(donor_path).convert('L'), dtype=np.uint8)
if donor.shape != C.shape:
    donor = np.array(Image.fromarray(donor, mode='L').resize((W, H), resample=Image.Resampling.BILINEAR), dtype=np.uint8)

CW_swap_external, tamper_external = replace_with_external_patch(
    CW_full,
    donor,
    dst_top=220, dst_left=280,
    src_top=200, src_left=200,
    h=110, w=110,
)
E_external, u_external = yeung_extract(CW_swap_external, mapping, Wr)

print(f'Donor: {donor_path}')
print(f'PSNR(CW_full, CW_swap_external): {psnr(CW_full, CW_swap_external):.4f} dB')
print(f'Fraction E=1 after external replacement: {E_external.mean():.6f}')

fig, ax = plt.subplots(1, 5, figsize=(21, 4))
ax[0].imshow(donor, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('Donor')
ax[0].axis('off')

ax[1].imshow(CW_swap_external, cmap='gray', vmin=0, vmax=255)
ax[1].set_title('CW after external replacement')
ax[1].axis('off')

ax[2].imshow(tamper_external, cmap='gray', vmin=0, vmax=1)
ax[2].set_title('Ground-truth tamper area')
ax[2].axis('off')

ax[3].imshow(E_external, cmap='gray', vmin=0, vmax=1)
ax[3].set_title('Extracted mask E')
ax[3].axis('off')

ax[4].imshow((E_external & tamper_external).astype(np.uint8), cmap='gray', vmin=0, vmax=1)
ax[4].set_title('Intersection: E & GT area')
ax[4].axis('off')

plt.tight_layout()
plt.show()


## Item 9. Analytical formulas for error diffusion (pull model)

Target bit at point $(n_1,n_2)$ is defined by tiled template:
$$
\tilde W(n_1,n_2)=W_r(n_1\bmod M_1,\,n_2\bmod M_2).
$$

Compensated intensity in pull model:
$$
\tilde C(n_1,n_2)=C(n_1,n_2)+\frac{7}{16}\varepsilon(n_1,n_2-1)+\frac{3}{16}\varepsilon(n_1-1,n_2+1)+\frac{5}{16}\varepsilon(n_1-1,n_2)+\frac{1}{16}\varepsilon(n_1-1,n_2-1).
$$

Nearest admissible value:
$$
C^W(n_1,n_2)=\arg\min_{x\in\{0,\dots,255\},\,\mu(x)=\tilde W(n_1,n_2)} |x-\tilde C(n_1,n_2)|.
$$

Local quantization error:
$$
\varepsilon(n_1,n_2)=\tilde C(n_1,n_2)-C^W(n_1,n_2).
$$


## Item 10. Analytical formulas for error diffusion (push model)

In push model, after selecting $C^W(n_1,n_2)$, local error
$$
\varepsilon(n_1,n_2)=\tilde C(n_1,n_2)-C^W(n_1,n_2)
$$
is distributed to not-yet-processed neighbors (Floyd-Steinberg kernel):

$$
\begin{aligned}
\tilde C(n_1,n_2+1) &\leftarrow \tilde C(n_1,n_2+1)+\frac{7}{16}\varepsilon(n_1,n_2),\\
\tilde C(n_1+1,n_2-1) &\leftarrow \tilde C(n_1+1,n_2-1)+\frac{3}{16}\varepsilon(n_1,n_2),\\
\tilde C(n_1+1,n_2) &\leftarrow \tilde C(n_1+1,n_2)+\frac{5}{16}\varepsilon(n_1,n_2),\\
\tilde C(n_1+1,n_2+1) &\leftarrow \tilde C(n_1+1,n_2+1)+\frac{1}{16}\varepsilon(n_1,n_2).
\end{aligned}
$$


## Item 11. Floyd-Steinberg embedding and comparison

Implement `yeung_embed_fs` (push model, Floyd-Steinberg kernel), then compare with previous variants (`simple`, `full`) using PSNR and BER.


In [ ]:
def yeung_embed_fs(C: np.ndarray, Wr: np.ndarray, mapping: np.ndarray) -> np.ndarray:
    """Yeung-Mintzer with Floyd-Steinberg error diffusion (push model)."""
    h, w = C.shape
    target = tile_wr(Wr, C.shape)

    work = C.astype(np.float64).copy()
    out = np.zeros_like(C, dtype=np.uint8)

    lower, upper = build_search_luts(mapping)

    for i in range(h):
        for j in range(w):
            bit = int(target[i, j])

            cur = int(np.clip(np.rint(work[i, j]), 0, 255))
            if mapping[cur] == bit:
                v = cur
            else:
                lo = int(lower[cur, bit])
                hi = int(upper[cur, bit])

                d_lo = abs(cur - lo) if lo >= 0 else 10_000
                d_hi = abs(hi - cur) if hi >= 0 else 10_000

                if d_hi < d_lo:
                    v = hi
                else:
                    v = lo if lo >= 0 else hi

                v = int(np.clip(v, 0, 255))

            out[i, j] = v
            err = float(work[i, j] - v)

            # Floyd-Steinberg: push error to not-yet-processed neighbors.
            if j + 1 < w:
                work[i, j + 1] += err * 7.0 / 16.0
            if i + 1 < h and j - 1 >= 0:
                work[i + 1, j - 1] += err * 3.0 / 16.0
            if i + 1 < h:
                work[i + 1, j] += err * 5.0 / 16.0
            if i + 1 < h and j + 1 < w:
                work[i + 1, j + 1] += err * 1.0 / 16.0

    return out


CW_fs = yeung_embed_fs(C, Wr, mapping)
E_fs, uCW_fs = yeung_extract(CW_fs, mapping, Wr)
expected = tile_wr(Wr, C.shape)

psnr_fs = psnr(C, CW_fs)
ber_fs = bit_error_rate(uCW_fs, expected)

summary = pd.DataFrame([
    {'method': 'simple (p3)', 'PSNR_dB': psnr(C, CW_simple), 'BER': bit_error_rate(yeung_extract_uCW(CW_simple, mapping), expected)},
    {'method': 'full nearest (p5)', 'PSNR_dB': psnr(C, CW_full), 'BER': bit_error_rate(yeung_extract_uCW(CW_full, mapping), expected)},
    {'method': 'Floyd-Steinberg (p11)', 'PSNR_dB': psnr_fs, 'BER': ber_fs},
])

print(summary.to_string(index=False))
print(f'Fraction of mismatches in E_fs: {E_fs.mean():.6f}')

fig, ax = plt.subplots(1, 4, figsize=(19, 4))
ax[0].imshow(CW_fs, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('CW (Floyd-Steinberg)')
ax[0].axis('off')

ax[1].imshow((CW_fs != C).astype(np.uint8), cmap='gray', vmin=0, vmax=1)
ax[1].set_title('Changed-pixel map')
ax[1].axis('off')

ax[2].imshow(np.abs(CW_fs.astype(np.int16) - C.astype(np.int16)), cmap='hot')
ax[2].set_title('|CW_fs - C|')
ax[2].axis('off')

ax[3].imshow(E_fs, cmap='gray', vmin=0, vmax=1)
ax[3].set_title('Mask E after extraction')
ax[3].axis('off')

plt.tight_layout()
plt.show()


# Теперь мой код